# Movie Recommendation Chatbot with Agno (TAG + RAG + Memory)

## Migrated from LangGraph → Agno

### Features:
- ✅ PDF document search (RAG) — Oscars 2026 PDF via FAISS vector store
- ✅ Genre classification (TAG) — Custom taxonomy from Excel
- ✅ Web search — Google Serper for live movie data
- ✅ TMDB / MovieLens access — Custom HTTP tools
- ✅ Use-case instructions — Loaded from Excel config
- ✅ Conversational memory — Agno built-in session storage
- ✅ All capabilities unified in a single Agno Agent

### Key Differences: LangGraph vs Agno
| Feature | LangGraph | Agno |
|---|---|---|
| Agent definition | StateGraph + nodes | `Agent(tools=[...])` |
| Memory | `MemorySaver` checkpointer | Built-in `session_id` |
| Tool binding | `llm.bind_tools()` | Pass list to `Agent` |
| Routing logic | `should_continue` function | Automatic |
| Execution | `app.stream()` | `agent.run()` / `agent.print_response()` |

## 1. Install Dependencies

In [ ]:
%pip install agno openai python-dotenv openpyxl pandas \
             google-search-results pypdf faiss-cpu \
             langchain-community langchain-openai requests --quiet
print("✓ All packages installed")

## 2. Imports & Environment

In [ ]:
import os
import json
import requests
import pandas as pd
from dotenv import load_dotenv

# Agno imports
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools import tool
from agno.memory.v2.db.sqlite import SqliteMemoryDb
from agno.memory.v2.memory import Memory
from agno.storage.sqlite import SqliteStorage

# RAG / embedding helpers (reusing LangChain loaders — Agno is model-agnostic)
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Web search
from langchain_community.utilities import GoogleSerperAPIWrapper

load_dotenv()
load_dotenv('../.env')

# Validate keys
for key in ["OPENAI_API_KEY", "SERPER_API_KEY"]:
    status = "✓" if os.getenv(key) else "✗ MISSING"
    print(f"{status} {key}")

TMDB_API_KEY = os.getenv("TMDB_API_KEY", "")
print(f"{'✓' if TMDB_API_KEY else '⚠ optional'} TMDB_API_KEY")

## 3. Helper Utilities
These replace the `ai_course_utils` helpers from the original notebook so the notebook is fully self-contained.

In [ ]:
# ── Load use-case config from Excel ────────────────────────────────────────
def load_use_case_config(filepath: str) -> dict:
    """
    Reads the use-case Excel file.
    Expected columns: 'key' and 'value'  (or first two columns are treated as such).
    Returns a dict, e.g. {'agent_prompt': '...', ...}
    """
    try:
        df = pd.read_excel(filepath)
        # Support both named columns and positional
        if 'key' in df.columns and 'value' in df.columns:
            return dict(zip(df['key'].astype(str), df['value'].astype(str)))
        else:
            cols = df.columns.tolist()
            return dict(zip(df[cols[0]].astype(str), df[cols[1]].astype(str)))
    except Exception as e:
        print(f"⚠ Could not load use-case config: {e}")
        return {}


# ── Load taxonomy from Excel ────────────────────────────────────────────────
def load_taxonomy_from_excel(filepath: str) -> dict:
    """
    Reads the taxonomy Excel.
    Expected columns: genre, included, guidelines  (first row = header).
    Returns {genre_name: {'included': '...', 'guidelines': '...'}, ...}
    """
    try:
        df = pd.read_excel(filepath)
        taxonomy = {}
        for _, row in df.iterrows():
            genre = str(row.get('genre', row.iloc[0]))
            taxonomy[genre] = {
                'included':   str(row.get('included',   row.iloc[1] if len(row) > 1 else '')),
                'guidelines': str(row.get('guidelines', row.iloc[2] if len(row) > 2 else '')),
            }
        return taxonomy
    except Exception as e:
        print(f"⚠ Could not load taxonomy: {e}")
        return {}


# ── Load PDF and build FAISS retriever ─────────────────────────────────────
def load_pdf_for_rag(filepath: str, chunk_size=1000, chunk_overlap=200):
    """
    Loads a PDF, splits into chunks, builds a FAISS vector store.
    Returns (documents, vectorstore, retriever).
    """
    loader = PyPDFLoader(filepath)
    pages  = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    docs = splitter.split_documents(pages)
    embeddings  = OpenAIEmbeddings()
    vectorstore = FAISS.from_documents(docs, embeddings)
    retriever   = vectorstore.as_retriever(search_kwargs={"k": 4})
    print(f"✓ PDF loaded: {len(pages)} pages → {len(docs)} chunks")
    return docs, vectorstore, retriever


print("✓ Utility functions defined")

## 4. Load Data

In [ ]:
# ── Use-case config ─────────────────────────────────────────────────────────
use_case_file = "../data/use_case_Movie_Recommendation.xlsx"
use_case_config = load_use_case_config(use_case_file)
system_prompt_from_config = use_case_config.get("agent_prompt", "")
print(f"✓ Use-case config loaded — agent_prompt length: {len(system_prompt_from_config)} chars")

# ── Taxonomy (TAG) ──────────────────────────────────────────────────────────
taxonomy_file = "../data/Movie_Recommendation_Taxonomy_File.xlsx"
taxonomy = load_taxonomy_from_excel(taxonomy_file)
print(f"✓ Taxonomy loaded — {len(taxonomy)} genres: {list(taxonomy.keys())}")

# ── PDF for RAG ─────────────────────────────────────────────────────────────
pdf_file = "../data/The_98th_Academy_Awards_2026.pdf"
documents, vectorstore, retriever = load_pdf_for_rag(pdf_file)

print(f"\n✅ All data ready!")

## 5. Define Tools

Each tool is a plain Python function decorated with Agno's `@tool` decorator.  
No `bind_tools()` call needed — just pass them to `Agent(tools=...)`.

In [ ]:
# ── Tool 1: Web Search (Google Serper) ──────────────────────────────────────
@tool(description="Search the web for current movie news, ratings, reviews, and showtimes.")
def search_web(query: str) -> str:
    """Search the web for current information about movies."""
    search = GoogleSerperAPIWrapper()
    return search.run(query)


# ── Tool 2: RAG — Search Oscars 2026 PDF ────────────────────────────────────
@tool(description="Search the 98th Academy Awards 2026 PDF for Oscar nominations and winners.")
def search_oscars_document(query: str) -> str:
    """Search the ingested Oscars 2026 PDF via semantic (vector) search."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant content found in the Oscars 2026 document."
    return "\n\n".join([d.page_content for d in docs])


# ── Tool 3: TAG — Genre Classification ─────────────────────────────────────
@tool(description="Classify a movie query into a custom genre using the taxonomy file (TAG pattern).")
def classify_genre(query: str) -> str:
    """Use the taxonomy Excel to classify user preference into a custom genre."""
    query_lower = query.lower()
    for genre_name, info in taxonomy.items():
        included = info.get('included', '').lower()
        keywords = [k.strip() for k in included.split(',') if k.strip()]
        if any(kw in query_lower for kw in keywords):
            return json.dumps({
                'genre': genre_name,
                'guidelines': info.get('guidelines', '')
            })
    return json.dumps({'genre': 'General', 'guidelines': 'No specific genre matched.'})


# ── Tool 4: TMDB Movie Search ────────────────────────────────────────────────
@tool(description="Search TMDB (The Movie Database) for detailed movie metadata, cast, and trailers.")
def search_tmdb(movie_title: str) -> str:
    """Fetch movie details from TMDB API."""
    if not TMDB_API_KEY:
        return "TMDB_API_KEY not configured. Set it in your .env file."
    url = "https://api.themoviedb.org/3/search/movie"
    params = {"api_key": TMDB_API_KEY, "query": movie_title}
    try:
        resp = requests.get(url, params=params, timeout=10)
        data = resp.json()
        results = data.get("results", [])[:3]
        if not results:
            return f"No TMDB results found for '{movie_title}'."
        out = []
        for m in results:
            out.append(
                f"Title: {m.get('title')}\n"
                f"Release: {m.get('release_date')}\n"
                f"Rating: {m.get('vote_average')}/10\n"
                f"Overview: {m.get('overview', '')[:300]}"
            )
        return "\n\n---\n\n".join(out)
    except Exception as e:
        return f"TMDB error: {e}"


# ── Tool 5: MovieLens Top Picks ──────────────────────────────────────────────
@tool(description="Fetch top-rated or popular movies from the MovieLens / TMDB popular endpoint.")
def get_popular_movies(genre_or_keyword: str = "popular") -> str:
    """Retrieve popular / top-rated movies — optionally filtered by keyword."""
    if not TMDB_API_KEY:
        return "TMDB_API_KEY not configured. Set it in your .env file."
    url = "https://api.themoviedb.org/3/movie/popular"
    params = {"api_key": TMDB_API_KEY, "language": "en-US", "page": 1}
    try:
        resp    = requests.get(url, params=params, timeout=10)
        results = resp.json().get("results", [])[:5]
        out = []
        for m in results:
            out.append(f"{m['title']} ({m.get('release_date','')[:4]}) — ⭐ {m.get('vote_average')}/10")
        return "Top Popular Movies:\n" + "\n".join(out)
    except Exception as e:
        return f"MovieLens/TMDB error: {e}"


ALL_TOOLS = [classify_genre, search_oscars_document, search_web, search_tmdb, get_popular_movies]
print(f"✓ {len(ALL_TOOLS)} tools defined: {[t.name for t in ALL_TOOLS]}")

## 6. Build the Agno Agent

### Architecture Comparison
```
LangGraph (before)              Agno (now)
─────────────────────────────   ──────────────────────────────────────
StateGraph(MessagesState)   →   Agent(model, tools, memory, storage)
add_node("agent", call_fn)  →   Handled internally by Agent
add_node("tools", ToolNode) →   Handled internally by Agent
conditional_edges(...)      →   Handled internally by Agent
MemorySaver checkpointer    →   SqliteStorage + Memory(db=SqliteMemoryDb)
app.stream({messages}, cfg) →   agent.run(message, session_id=...)
```

In [ ]:
# ── System prompt ────────────────────────────────────────────────────────────
# Use the prompt loaded from the Excel config; fall back to built-in if blank.
AGENT_SYSTEM_PROMPT = system_prompt_from_config or """
You are an expert Movie Recommendation Assistant with five specialised tools:

1. **classify_genre**       – TAG: Identify the user's preferred genre from the custom taxonomy.
2. **search_oscars_document** – RAG: Search the 98th Academy Awards 2026 PDF for nominations/winners.
3. **search_web**           – Live web search (Google Serper) for current movie news and reviews.
4. **search_tmdb**          – TMDB API: Detailed metadata, cast, ratings for specific movie titles.
5. **get_popular_movies**   – TMDB Popular endpoint (MovieLens-style top picks).

### Recommended Tool Call Sequence:
1. Always call **classify_genre** first to understand the user's taste.
2. If the question involves Oscars or awards → call **search_oscars_document**.
3. For a specific title → call **search_tmdb**.
4. For current news, release dates, streaming → call **search_web**.
5. For general trending suggestions → call **get_popular_movies**.

Always provide structured, helpful responses with movie titles, ratings, and brief explanations.
Maintain context across the conversation — remember what genres and movies the user mentioned.
""".strip()

print(f"✓ System prompt ready ({len(AGENT_SYSTEM_PROMPT)} chars)")

In [ ]:
# ── Persistent memory & session storage ─────────────────────────────────────
# SqliteMemoryDb stores cross-session user facts (replaces MemorySaver)
memory_db = SqliteMemoryDb(table_name="movie_bot_memory", db_file="movie_bot.db")
memory    = Memory(db=memory_db)

# SqliteStorage keeps full conversation history per session_id
storage = SqliteStorage(table_name="movie_bot_sessions", db_file="movie_bot.db")

# ── Agno Agent (replaces the entire LangGraph StateGraph) ───────────────────
movie_agent = Agent(
    name="MovieRecommendationBot",
    model=OpenAIChat(id="gpt-4o"),          # swap to gpt-4o-mini for cost savings
    tools=ALL_TOOLS,
    instructions=AGENT_SYSTEM_PROMPT,
    memory=memory,
    storage=storage,
    enable_agentic_memory=True,             # auto-save important user facts
    add_history_to_messages=True,           # full conversational context
    num_history_responses=10,               # last N exchanges
    show_tool_calls=True,                   # print tool calls (set False in prod)
    markdown=True,
)

print("✅ Agno Movie Recommendation Agent ready!")
print(f"   Model  : {movie_agent.model.id}")
print(f"   Tools  : {[t.name for t in ALL_TOOLS]}")
print(f"   Memory : SqliteMemoryDb  → movie_bot.db")
print(f"   Storage: SqliteStorage  → movie_bot.db")

## 7. Chat Helper Function

A thin wrapper so the interface matches the original `chat()` function.

In [ ]:
def chat(user_input: str, session_id: str = "default") -> str:
    """
    Send a message to the Agno Movie Agent and return the reply as a string.

    Parameters
    ----------
    user_input : str   — The user's message.
    session_id : str   — Conversation thread ID (replaces LangGraph thread_id).

    Returns
    -------
    str — The agent's reply.
    """
    response = movie_agent.run(user_input, session_id=session_id)
    return response.content


print("🎬 chat() helper ready — usage: chat('What sci-fi movies won Oscars?')")

## 8. Test Conversations

The test dialogue mirrors the original LangGraph notebook exactly.

In [ ]:
# ── Conversation 1: TAG + RAG + memory ──────────────────────────────────────
session = "oscars_test"

q1 = "I love mythology mixed with sci-fi. Are there Oscar nominees like that?"
print(f"User: {q1}")
r1 = chat(q1, session_id=session)
print(f"Bot : {r1[:300]}...\n")

q2 = "Tell me more about the first one you mentioned"
print(f"User: {q2}")
r2 = chat(q2, session_id=session)
print(f"Bot : {r2[:300]}...\n")

q3 = "When are the Oscars?"
print(f"User: {q3}")
r3 = chat(q3, session_id=session)
print(f"Bot : {r3[:300]}...")

In [ ]:
# ── Conversation 2: TMDB tool ────────────────────────────────────────────────
session2 = "tmdb_test"

q4 = "What can you tell me about Dune Part Two on TMDB?"
print(f"User: {q4}")
r4 = chat(q4, session_id=session2)
print(f"Bot : {r4[:400]}...")

In [ ]:
# ── Conversation 3: Popular movies (MovieLens-style) ─────────────────────────
session3 = "popular_test"

q5 = "What are the most popular movies right now?"
print(f"User: {q5}")
r5 = chat(q5, session_id=session3)
print(f"Bot : {r5[:400]}...")

In [ ]:
# ── Conversation 4: Full pipeline — genre + awards + web ─────────────────────
session4 = "full_pipeline"

questions = [
    "I enjoy dark psychological thrillers. Any 2025 recommendations?",
    "Did any of those get Oscar nominations this year?",
    "What do critics say about the best one?",
]

for q in questions:
    print(f"\nUser: {q}")
    r = chat(q, session_id=session4)
    print(f"Bot : {r[:350]}...")

## 9. Interactive Chat Loop (Optional)

Run this cell for a live interactive session in Jupyter.

In [ ]:
# Uncomment to start an interactive chat loop

# session_id = "interactive"
# print("🎬 Movie Recommendation Bot (type 'quit' to exit)\n")
# while True:
#     user_input = input("You: ").strip()
#     if user_input.lower() in ("quit", "exit", "bye"):
#         print("Bye! Enjoy your movies 🍿")
#         break
#     if not user_input:
#         continue
#     response = chat(user_input, session_id=session_id)
#     print(f"\nBot: {response}\n")

## 10. Architecture Summary

```
┌─────────────────────────────────────────────────────────────────┐
│                   Agno Movie Recommendation Agent               │
│                                                                 │
│  ┌───────────┐   ┌────────────────────────────────────────────┐ │
│  │  OpenAI   │   │               Tools                        │ │
│  │  GPT-4o   │◄──┤  1. classify_genre      (TAG - Excel)      │ │
│  └─────┬─────┘   │  2. search_oscars_doc   (RAG - PDF+FAISS)  │ │
│        │         │  3. search_web          (Google Serper)    │ │
│        │         │  4. search_tmdb         (TMDB API)         │ │
│        │         │  5. get_popular_movies  (TMDB Popular)     │ │
│        │         └────────────────────────────────────────────┘ │
│        │                                                         │
│  ┌─────▼────────────────────────────────────────────────────┐   │
│  │  Memory & Storage (SQLite)                               │   │
│  │  • SqliteMemoryDb  — persistent user facts               │   │
│  │  • SqliteStorage   — full session history per thread_id  │   │
│  └──────────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────────┘

Data Sources
  ├── use_case_Movie_Recommendation.xlsx  → agent instructions
  ├── Movie_Recommendation_Taxonomy_File.xlsx → genre keywords (TAG)
  └── The_98th_Academy_Awards_2026.pdf    → vector chunks (RAG)
```

### Why Agno over LangGraph for this use case?
- **Less boilerplate**: No StateGraph, no node wiring, no conditional edges — just `Agent(tools=...)`.
- **Built-in memory tiers**: Session storage and agentic long-term memory out of the box.
- **Same tool interface**: Plain Python functions with a decorator — no LangChain `@tool` dependency.
- **Easy model switching**: Change `OpenAIChat(id=...)` to `AnthropicChat`, `Groq`, etc. in one line.
- **Production-ready**: Agno agents run as FastAPI services with `agent.as_app()` — no extra code.